### Загрузка и обработка данных

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from Modules.panel_utils import (
    detect_outlier_regions_iqr,
    run_panel_unit_root_tests,
)
from Modules.seasonal_adjustment import (
    run_friedman_seasonality_tests,
)


In [34]:
###############################
    #ЗАГРУЗКА + ПРЕДОБРАБОТКА ДАННЫХ
###############################
df_RF = pd.read_excel('База данных_рег и фед показатели.xlsx', skiprows=1, sheet_name = 'data_RF')
df_reg = pd.read_excel('База данных_рег и фед показатели.xlsx',skiprows=1 , sheet_name = 'data')
households_structure = pd.read_excel('База_соц_эконом_неоднор_регионов.xlsx', skiprows=1, sheet_name = 'Фин пол ДХ')
households_loans = pd.read_excel('База_соц_эконом_неоднор_регионов.xlsx', sheet_name = 'КЖН (рейтинг)')
funds_coefficient = pd.read_excel('База_соц_эконом_неоднор_регионов.xlsx', skiprows=2, sheet_name = 'Коэф фондов')
pickle_filename = 'region_cluster_dataset.pkl'

with open(pickle_filename, 'rb') as f:
    df_region_cluster = pickle.load(f)

# Преобразование даты 
df_RF['Date'] = pd.to_datetime(df_RF['Date'])
df_RF = df_RF.sort_values('Date')
df_reg['Date'] = pd.to_datetime(df_reg['Date'])
df_reg = df_reg.sort_values(['Num_reg','Date'])


In [35]:
###############################
# АНАЛИЗ ПАНЕЛЬНЫХ ДАННЫХ
###############################
pickle_filename = 'mon_shock_dataset.pkl'
with open(pickle_filename, 'rb') as f:
    df_shocks = pickle.load(f)
df_shocks['Date'] = pd.to_datetime(df_shocks['Date'])

if isinstance(df_reg.index, pd.MultiIndex):
    df_reg = df_reg.reset_index()

df_reg['Date'] = pd.to_datetime(df_reg['Date'])

#  ДОБАВЛЯЕМ КЛАСТЕРЫ 
df_reg = df_reg.merge(df_region_cluster, on='Region', how='left')

# ============ ФЕДЕРАЛЬНЫЕ ПЕРЕМЕННЫЕ ============
df_reg = df_reg.merge(df_shocks[['Date', 'Mon_Shock']], on='Date', how='left')

df_RF.rename(columns={'exc_rate': 'Exc_rate'}, inplace=True)
df_RF.rename(columns={'exc_rate_diff': 'd_Ex_Rate'}, inplace=True)

df_rf_extra = df_RF[['Date', 'ROISFIX', 'MIACR', 'Covid_dum', 'Sank_dum',
                'Exc_rate', 'Bonds_Rate_Correct_5Y', 'Inflation_Expectations',
                'MaP_Announcement', 'MaP_Tight_Announcement', 'MaP_Ease_Announcement',
                'MaP_Fact','MaP_Tight_Fact', 'MaP_Ease_Fact',]].copy()
df_reg = df_reg.merge(df_rf_extra, on='Date', how='left')
df_reg.rename(columns={'CPI': 'CPI_reg'}, inplace=True)

map_cols = [
    'MaP_Tight_Announcement', 
    'MaP_Ease_Announcement', 
    'MaP_Tight_Fact', 
    'MaP_Ease_Fact'
]
df_RF[map_cols] = df_RF[map_cols].fillna(0)

### Преобразование данных

In [36]:
# Сортировка
df_reg = df_reg.sort_values(['Region', 'Date']).copy()
df_RF = df_RF.sort_values(['Date']).copy()

# ============ ЛОГАРИФМИРОВАНИЕ ============ #

loan_vars = ['New_Loans_Fl', 'New_Loans_Mort', 'New_Loans_ConsCred', 'Fin_Dostup']

for var in loan_vars:
    if var in df_reg.columns:
        df_reg[f'ln_{var}'] = np.log(df_reg[var])
        
ln_loan_vars = [f'ln_{var}' for var in loan_vars if f'ln_{var}' in df_reg.columns]


# ============ СОЗДАНИЕ ПЕРВЫХ РАЗНОСТЕЙ  ============ #

# Разности для логарифмов 
for var in ln_loan_vars:
    d_var = f'd_{var}'
    df_reg[d_var] = df_reg.groupby('Region')[var].diff()

################################
# Разности переменных регионов #
################################

diff_vars_reg = ['Int_Rate_FL', 'Int_Rate_Mort', 'Int_Rate_ConsCred', \
            'Mon_Shock', 'ROISFIX', 'MIACR','CPI_reg']

for var in diff_vars_reg:
    d_var = f'd_{var}'
    df_reg[d_var] = df_reg.groupby('Region')[var].diff()

#################################
# Разности федеральных переменных
#################################

df_RF = df_RF.sort_values(['Date'])

diff_vars_fed = ['CPI', 'Key_Rate', 'ROISFIX', 'MIACR',
                'IBC_constr_fed', 'IBC_torg_fed', 'IBC_auto_fed',]

for var in diff_vars_fed:
    d_var = f'd_{var}'
    df_RF[d_var] = df_RF[var].diff()


# Разделим шоки на позитивные и негативные
# Mon_Shock
df_reg['d_Mon_Shock_neg'] = df_reg['d_Mon_Shock'].where(df_reg['d_Mon_Shock'] < 0, 0)
df_reg['d_Mon_Shock_pos'] = df_reg['d_Mon_Shock'].where(df_reg['d_Mon_Shock'] > 0, 0)
# Разность ROISFIX 
df_reg['d_ROISFIX_neg'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] < 0, 0)
df_reg['d_ROISFIX_pos'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] > 0, 0)
# Разность MIACR 
df_reg['d_MIACR_neg'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] < 0, 0)
df_reg['d_MIACR_pos'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] > 0, 0)


#  Создаем взаимодействия для переменных процентных ставок
map_vars_int = ['d_Mon_Shock',
    'd_Mon_Shock_neg',
    'd_Mon_Shock_pos',
    'd_ROISFIX',
    'd_ROISFIX_neg',
    'd_ROISFIX_pos',
    'd_MIACR',
    'd_MIACR_neg',
    'd_MIACR_pos',
]

z_vars_int = [
    'Covid_dum',
    'Sank_dum',
    'Cluster_1',
    'Cluster_2'
]
for m in map_vars_int:
    for z in z_vars_int:
        col_name = f"{m}_{z}"
        df_reg[col_name] = df_reg[m] * df_reg[z]


#  Создаем взаимодействия для переменных макропру
map_var_makro = [
    'MaP_Tight_Announcement',
    'MaP_Ease_Announcement',
    'MaP_Tight_Fact',
    'MaP_Ease_Fact',
]

df_reg['D_top5_rozn_lag1'] = df_reg.groupby('Region')['D_top5_rozn'].shift(1)
df_reg['Cap_to_assets_lag1'] = df_reg.groupby('Region')['Cap_to_assets'].shift(1)

z_vars_makro = [
    'D_top5_rozn_lag1',
    'Cap_to_assets_lag1',
    'Cluster_1',
    'Cluster_2'
]
for m in map_var_makro:
    for z in z_vars_makro:
        col_name = f"{m}_{z}"
        df_reg[col_name] = df_reg[m] * df_reg[z]


In [37]:
# СПИСОК ПЕРЕМЕННЫХ
all_vars = {
'conscred_vars' : [
    'd_Int_Rate_ConsCred',          # Зависимая переменная
    'Int_Rate_ConsCred',
    'New_Loans_ConsCred',           # Выдача
    'New_Loans_ConsCred_mom',
    'New_Loans_Fl',
    'New_Loans_Fl_mom',
    'Zadolg_ConsCred',              # Задолженность
    'Zadolg_ConsCred_mom', 
    'Zadolg_Fl',
    'Zadolg_Fl_mom',
    'Def_Zadolg_ConsCred',          # Дефицит задолж
    'Def_Zadolg_Fl',
    'Cred_nagr',                    # Переменные состояния финансового рынка
    'Fin_Dostup',
    'D_top5_rozn',
    'Zakred',
    'Exc_rate',                     # Переменные ожиданий
    'Bonds_Rate_Correct_5Y',
    'Inflation_Expectations',
    #
    'Mon_Shock',                    # Шоки
    'd_Mon_Shock',
    'd_Mon_Shock_neg',
    'd_Mon_Shock_pos',
    'ROISFIX',
    'MIACR',
    'Covid_dum',                    # Дамми
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
],

'other_vars' : [
    'Int_Rate_FL',
    'Int_Rate_FL_lag1',
    'Int_Rate_Mort',
    'Credit_impulse',
    'Int_Rate_Mort_lag1',
    'Int_Rate_ConsCred',
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Cred_structure',
    'Def_Zadolg_Fl',
    'Def_Zadolg_Mort',
    'Def_Zadolg_ConsCred',
    'Mon_Shock',
    'Exc_rate',
    'Inflation_Expectations',
    'Bonds_Rate_Correct_5Y',
    'Cluster_1',
    'Cluster_2',
    'New_Loans_Progr',
    'MIACR',
    'Covid_dum',
    'Sank_dum',
    'ROISFIX'
]
}
choosed_vars = all_vars['conscred_vars']
name = [key for key, value in all_vars.items() if value == choosed_vars][0][:-5]

df_reg_analys = df_reg[['Region', 'Date'] + choosed_vars].copy()
df_reg_analys = df_reg_analys[df_reg_analys['Region'] != 'Москва']
cols_for_drop = [c for c in df_reg_analys.columns if c not in ['New_Loans_Progr']]
df_reg_analys = df_reg_analys.dropna(subset=cols_for_drop)

### IQR чистка

In [38]:
print("\n" + "="*70)
print("ОЧИСТКА РЕГИОНОВ ПО МЕТОДУ IQR")
print("="*70)

key_vars = [x for x in choosed_vars if x not in [

    'Mon_Shock',                    # Шоки
    'd_Mon_Shock',
    'Mon_Shock_neg',
    'Mon_Shock_pos',
    'd_Mon_Shock_neg',
    'd_Mon_Shock_pos',
    'ROISFIX',
    'MIACR',
    'Cluster_1',
    'Cluster_2',
    'Covid_dum',
    'Sank_dum',
    
    'd_IBC_constr',
    'd_IBC_torg',
    'd_IBC_auto',
    ]
]


# АНАЛИЗ РЕГИОНОВ ПО IQR
region_quality_iqr = detect_outlier_regions_iqr(df_reg_analys, key_vars, min_obs=60, max_na_pct=0.3)

print(f"\n АНАЛИЗ {len(region_quality_iqr)} РЕГИОНОВ (IQR)")
print(region_quality_iqr.to_string(index=False))

# СТАТИСТИКА
exclude_count = len(region_quality_iqr[region_quality_iqr['Status'] == 'EXCLUDE'])
keep_count = len(region_quality_iqr) - exclude_count

print(f"\n СТАТИСТИКА:")
print(f"Всего регионов: {len(region_quality_iqr)}")
print(f"Исключено: {exclude_count} ({exclude_count/len(region_quality_iqr)*100:.1f}%)")
print(f"Оставлено: {keep_count}")

# СПИСОК ХОРОШИХ РЕГИОНОВ
good_regions_iqr = region_quality_iqr[region_quality_iqr['Status'] == 'KEEP']['Region'].tolist()

# СОХРАНЕНИЕ СПИСКА РЕГИОНОВ
with Path('filter_for_cluster.pkl').open("wb") as saved:
    pickle.dump(good_regions_iqr, saved)

# ФИЛЬТРАЦИЯ ДАННЫХ
df_reg_analys= df_reg_analys[df_reg_analys['Region'].isin(good_regions_iqr)].copy()
print(f"\n Форма после очистки (IQR): {df_reg_analys.shape}")


ОЧИСТКА РЕГИОНОВ ПО МЕТОДУ IQR

 АНАЛИЗ 77 РЕГИОНОВ (IQR)
                             Region  N_obs NA_pct  IQR_outliers IQR_pct Reasons Status
                     Алтайский край     77   0.0%             9   11.7%           KEEP
                   Амурская область     77   0.0%            15   19.5%           KEEP
              Архангельская область     77   0.0%            10   13.0%           KEEP
               Астраханская область     77   0.0%            10   13.0%           KEEP
               Белгородская область     77   0.0%             6    7.8%           KEEP
                   Брянская область     77   0.0%             7    9.1%           KEEP
               Владимирская область     77   0.0%             5    6.5%           KEEP
              Волгоградская область     77   0.0%             8   10.4%           KEEP
                Вологодская область     77   0.0%             8   10.4%           KEEP
                Воронежская область     77   0.0%             6    7.8%

### Тест Фридмана на сезонность

In [40]:
friedman_summary = run_friedman_seasonality_tests(df_RF, date_col='Date')


РЕЗУЛЬТАТЫ ТЕСТОВ ФРИДМАНА
                  Variable  Chi_Squared      P_Value  Has_Seasonality  N_Years  N_Months
                   Num_reg          NaN          NaN            False        5        12
             CAR_Indicator    12.107692 3.556030e-01            False        5        12
              D_Progr_mort    11.707692 3.860113e-01            False        5        12
                   ROISFIX     4.456963 9.545978e-01            False        5        12
                  Exc_rate    15.276923 1.701670e-01            False        5        12
                 d_Ex_Rate    17.892308 8.411333e-02            False        5        12
     Bonds_Rate_Correct_5Y     3.800000 9.754064e-01            False        5        12
    Inflation_Expectations    21.442254 2.906820e-02             True        5        12
      Nominal_Percent_Rate    11.863422 3.739965e-01            False        5        12
       Ent_conf_ind_mining    21.098503 3.236478e-02             True        5    

c:\Users\donat\anaconda3\envs\Credit\Lib\site-packages\scipy\stats\_stats_py.py:8780: RuntimeWarning: invalid value encountered in scalar divide
  statistic = (12.0 / (k*n*(k+1)) * ssbn - 3*n*(k+1)) / c


In [ ]:
# # Extra diagnostics after Friedman test
# from Modules.seasonal_adjustment import run_seasonal_diagnostics

# series_list = [
#     'Inflation_Expectations',
#     'Ent_conf_ind_mining',
#     'Ent_conf_ind_manufactoring'
# ]

# for col in series_list:
#     if col in df_RF.columns:
#         run_seasonal_diagnostics(df_RF, 'Date', col, shock_year="2022")
#     else:
#         print(f'Missing column: {col}')


# # ####################################
# # # Сезонность есть даже с учётом шока в: Inflation_Expectations, Ent_conf_ind_manufactoring

### ===== СЕЗОННАЯ КОРРЕКТИРОВКА ДЛЯ ПАНЕЛЬНЫХ ДАННЫХ =====

In [ ]:
df_reg_analys.columns

In [ ]:
# Сезонная корректировка 
# ! Для корректировки НЕОБХОДИМО добавить новые переменные в список функции seasonal_adjust_panel_wrapper в seasonal_adjustment.py
# 'Inflation_Expectations', 'Ent_conf_ind_manufactoring', 'Ent_conf_ind_mining' потенциальные переменные
# Output_Index, IBC_constr, IBC_torg, IBC_auto, IBC_constr_fed, IBC_torg_fed, IBC_auto_fed

from Modules.seasonal_adjustment import seasonal_adjust_panel_wrapper

col_name_reg = ['Inflation_Expectations']  
df_reg_analys = seasonal_adjust_panel_wrapper(df_reg_analys, col_name_reg)

col_name_fed = ['d_IBC_constr_fed', 'd_IBC_torg_fed', 'd_IBC_auto_fed'] 
df_RF = seasonal_adjust_panel_wrapper(df_RF, col_name_fed)


SyntaxError: invalid syntax (966468872.py, line 6)

In [42]:
# Чистка датафрейма
to_delete_columns_reg = [col for col in df_reg_analys.columns if col.endswith('_seasonal') or col.endswith('_trend')]
to_delete_columns_RF = [col for col in df_RF.columns if col.endswith('_seasonal') or col.endswith('_trend')]

df_reg_analys.drop(columns=to_delete_columns_reg, inplace=True)
df_RF.drop(columns=to_delete_columns_RF, inplace=True)

# Разность инфляционных ожиданий
df_reg_analys['d_Inflation_Expectations'] = df_reg_analys['Inflation_Expectations_adj'].diff()

KeyError: 'Inflation_Expectations_adj'

### Tестирование единичных корней

In [ ]:
# # ===== ЗАПУСК ТЕСТОВ НА КОНКРЕТНЫЕ ПЕРЕМЕННЫЕ =====

# print("\n" + "="*70)
# print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ - КЛАСТЕР 1")
# print("="*70)

# # === СПИСОК ПЕРЕМЕННЫХ ДЛЯ ТЕСТИРОВАНИЯ ===
# custom_vars = [x for x in choosed_vars if x not in [
#     'Cluster_1',
#     'Cluster_2',
#     'Covid_dum',
#     'Sank_dum']
# ]

# df_reg_analys_clus_1 = df_reg_analys[df_reg_analys['Cluster_1'] == 1].copy()

# # Запуск функции с пользовательским списком переменных
# results_seasonal_adjusted = run_panel_unit_root_tests(df_reg_analys_clus_1, panel_vars=custom_vars)

In [ ]:
# print("\n" + "="*70)
# print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ ДЛЯ НАЧАЛЬНЫХ ПЕРЕМЕННЫХ - КЛАСТЕР 2")
# print("="*70)


# df_reg_analys_clus_two = df_reg_analys[df_reg_analys['Cluster_2'] == 1].copy()

# # Запуск функции с пользовательским списком переменных
# results_seasonal_adjusted = run_panel_unit_root_tests(df_reg_analys_clus_two, panel_vars=custom_vars)

In [ ]:
# print("\n" + "="*70)
# print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ ДЛЯ НАЧАЛЬНЫХ ПЕРЕМЕННЫХ - КЛАСТЕР 3")
# print("="*70)

# df_reg_analys_clus_three = df_reg_analys[(df_reg_analys['Cluster_1'] == 0) & (df_reg_analys['Cluster_2'] == 0)].copy()

# # Запуск функции с пользовательским списком переменных
# results_seasonal_adjusted = run_panel_unit_root_tests(df_reg_analys_clus_three, panel_vars=custom_vars)

In [ ]:
# print("\n" + "="*70)
# print("ЗАПУСК ТЕСТОВ НА ЕДИНИЧНЫЕ КОРНИ ДЛЯ НАЧАЛЬНЫХ ПЕРЕМЕННЫХ - ВСЕ ДАННЫЕ")
# print("="*70)
# # Запуск функции с пользовательским списком переменных
# results_seasonal_adjusted_all = run_panel_unit_root_tests(df_reg_analys, panel_vars=custom_vars)

### Сохранение результатов

In [ ]:
df_reg_analys.to_excel(f'Operations/{name}_reg_analys.xlsx')
df_RF.to_excel('Operations/fed_analys.xlsx')